# PALLAS surrogate models — kickoff notebook

**This is the only notebook you need to start.** It answers three questions before you write a
line of model code — what one simulation of this machine takes **in**, what it puts **out**, and
what sits in **every file** of the pack — and then it builds two scored baselines and writes two
valid submissions. About forty minutes end to end, all of it on a laptop CPU.

PALLAS is a **laser-plasma accelerator**: a laser pulse drives a plasma wave in a
few-millimetre gas cell, and electrons stripped from the nitrogen inner shell ride that wave
to between roughly 40 and 500 MeV in a very short distance. The operator sets a handful of
knobs and gets one electron bunch out. Everything here is simulation, which is deliberate —
it gives every sample an exact ground-truth label, which no measurement does.

Sections 1 to 5 are the tour: nothing is scored, each file is opened and its shape printed.
Sections 6 to 9 are the work: the columns, the two conventions, and a baseline in each
direction — the beam read back to the settings, and the settings read forward to the beam.

**`objectives.md`** lists what is actually being asked, easy to hard. **`scientific_case.md`**
explains why the machine matters. **`data/README.md`** gives the splits and the scoring rules
in full.


## Before you start

You need the data pack, which is **not in this repository** — see [`data/README.md`](data/README.md)
for the download and unpack it as `data/pallas_hackathon_data/`. Then:

```
pip install -r requirements.txt
```

Everything in this notebook runs on a laptop CPU in well under an hour.


## The pack — find it once

Set `PALLAS_PACK` if you unpacked the data somewhere else. Every cell below reads from `PACK`,
and from the two frames this cell loads. There is no reloading later in the notebook.


In [1]:
import json
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# The pack is not in this folder: download it and unpack it under data/.
# Set PALLAS_PACK if you keep it anywhere else.
CANDIDATES = [os.environ.get("PALLAS_PACK"),
              os.path.join("data", "pallas_hackathon_data"),
              os.path.join("..", "pallas_hackathon_data"),
              "pallas_hackathon_data"]
PACK = next((p for p in CANDIDATES
             if p and os.path.exists(os.path.join(p, "campaign_A.parquet"))), None)
if PACK is None:
    raise SystemExit("no data pack found -- see data/README.md for the download "
                     "and unpack it as data/pallas_hackathon_data/")

n_files = sum(1 for f in os.listdir(PACK) if os.path.isfile(os.path.join(PACK, f)))
size_mb = sum(os.path.getsize(os.path.join(PACK, f)) for f in os.listdir(PACK)
              if os.path.isfile(os.path.join(PACK, f))) / 1e6
print(f"pack at {PACK}: {n_files} files, {size_mb:.0f} MB\n")

# The array columns (spectra, density profiles) are most of the file size, so ask
# for the scalars unless you need them. Both campaigns are loaded once, here.
SCALARS = ["scan", "config", "draw", "sub_scan", "row_id", "p_1", "a_0", "c_N2", "x_of",
           "P_max", "cN2_max", "L_inj", "dip_frac",
           "E_med_MeV", "E_mean_MeV", "dE_mad", "dE_std", "q_end", "i_peak",
           "sigma_z", "sigma_y", "n_emit_x", "n_emit_y", "div_rms", "beta_x",
           "alpha_x", "z_avg", "injected"]


def load(name):
    path = f"{PACK}/{name}.parquet"
    cols = pq.ParquetFile(path).schema.names      # the footer, not the data
    return pd.read_parquet(path, columns=[c for c in SCALARS if c in cols])


# Two campaigns, two files. Campaign A holds both of its scans, `scan` says which.
A = load("campaign_A")
B = load("campaign_B")
grid = np.load(f"{PACK}/common_energy_grid_MeV.npy")   # shared energy axis, MeV


def summarise(label, frame):
    print(f"{label:22s} {len(frame):6,} rows   {frame.injected.sum():5,} injected"
          f"   ({frame.injected.mean():.1%})")


summarise("campaign_A", A)
for scan, rows in A.groupby("scan"):
    summarise(f"   scan = {scan}", rows)
summarise("campaign_B", B)


pack at data/pallas_hackathon_data: 12 files, 188 MB

campaign_A             13,604 rows   11,201 injected   (82.3%)
   scan = random        9,816 rows   7,658 injected   (78.0%)
   scan = uniform       3,788 rows   3,543 injected   (93.5%)
campaign_B              4,886 rows   2,482 injected   (50.8%)


## 1. What one simulation is

PALLAS is a laser-plasma accelerator. A high-power laser pulse is focused into a
few-millimetre cell of helium mixed with a little nitrogen. The pulse drives a wave in the
plasma behind it; electrons stripped from the inner shell of the nitrogen fall into that wave
and are accelerated to somewhere between roughly 40 and 500 MeV in a few millimetres. A
conventional accelerator needs tens of metres to do the same thing.

**One simulation is one setting of the knobs, run start to finish, producing one electron
bunch.** It is a particle-in-cell simulation run with Smilei: the code follows the laser
field, the plasma and the electrons through the cell on a grid, so the bunch that comes out
is computed from first principles rather than fitted. That is the reason for using simulation
here at all — every sample carries an exact ground-truth label, which no measurement does.

The pack holds two **campaigns**. They are two separate studies of the same machine and they
do not share a knob set, which is why they are two files.

### The inputs — everything that is chosen before a simulation runs

**Campaign A** varies four scalars: the gas, the nitrogen, and where and how hard the laser
is focused.

| input | meaning | unit |
|---|---|---|
| `p_1` | pressure in the injection region — how much gas there is | mbar |
| `a_0` | laser amplitude, the normalised vector potential — how hard the laser drives the plasma | dimensionless |
| `c_N2` | nitrogen concentration in the helium — the atoms the accelerated electrons come from | fraction |
| `x_of` | focal position — where along the cell the laser is focused | µm |

**Campaign B** fixes the laser and varies the **shape of the gas density along the cell**
instead. Its four knobs are the four numbers that describe that shape.

| input | meaning | unit |
|---|---|---|
| `P_max` | peak pressure of the density profile | Pa |
| `cN2_max` | peak nitrogen fraction | fraction |
| `L_inj` | length of the injection region | mm |
| `dip_frac` | depth of the dip in the density that triggers injection | fraction |
| `x_of` | focal position, as in Campaign A | µm |

Campaign B also ships **the density profile itself** as a 2000-point curve (`x_p`, `n_e_p`) —
the four knobs are a summary of that curve, and the curve is the thing the simulation was
actually given. Campaign A ships a profile too, but as only 6 points, because its gas cell is
described by a handful of corner positions rather than a sampled curve.

**`x_p` is in micrometres, not metres.** The cell below checks it rather than asserting it:
the curve ends exactly at `L_inj + 3.2` mm, the plasma end every trajectory score in this
pack is split at. `n_e_p` is in m⁻³.

**Held fixed in both, so it is in no column:** the cell geometry, the laser pulse duration and
wavelength, the helium, and every numerical setting of the simulation. Campaign B additionally
fixes `a_0`.

In [2]:
A_KNOBS = ["p_1", "a_0", "c_N2", "x_of"]
B_KNOBS = ["P_max", "cN2_max", "L_inj", "dip_frac", "x_of"]

b0 = B.index[B.injected][0]          # one row that produced a bunch, used all the way down
cfg = int(B.loc[b0, "config"])       # its simulation id

print("one Campaign A simulation was asked for:")
print(A.loc[A.index[A.injected][0], A_KNOBS].to_string(), "\n")
print(f"one Campaign B simulation was asked for (config {cfg}):")
print(B.loc[b0, B_KNOBS].to_string(), "\n")

prof = pd.read_parquet(f"{PACK}/campaign_B.parquet", columns=["x_p", "n_e_p"]).loc[b0]
x_p = np.asarray(prof.x_p)           # position along the cell, MICROmetres
n_e_p = np.asarray(prof.n_e_p)       # electron density there, m^-3
print(f"and it was given a density curve of {x_p.size} points: "
      f"{x_p.min() / 1e3:.3f} to {x_p.max() / 1e3:.3f} mm, "
      f"peak density {n_e_p.max():.3e} m^-3")
print(f"  that curve ends at L_inj + 3.2 mm = "
      f"{B.loc[b0, 'L_inj'] + 3.2:.3f} mm, which is the plasma end this pack scores against")

one Campaign A simulation was asked for:
p_1       90.25957
a_0       1.326063
c_N2      0.053635
x_of    845.478179 

one Campaign B simulation was asked for (config 1163):
P_max       5847.603101
cN2_max         0.21759
L_inj          0.454187
dip_frac       0.266643
x_of         290.397487 



and it was given a density curve of 2000 points: 0.000 to 3.654 mm, peak density 2.638e+24 m^-3
  that curve ends at L_inj + 3.2 mm = 3.654 mm, which is the plasma end this pack scores against


### The outputs — what comes out, at four levels of detail

The same simulation is stored four times over, each time in more detail. Which one you want
decides which file you open, and it is the single most useful thing to know about this pack.

| level | what it is | where | size of one simulation |
|---|---|---|---|
| 1 | the finished bunch as **a handful of numbers** — energy, charge, spread, length, emittance, divergence | `campaign_A.parquet`, `campaign_B.parquet` | ~20 numbers |
| 2 | its **energy spectrum** — how much charge at each energy | the `spec_common` column of the same files | 200 bins |
| 3 | the bunch **along the accelerator**, not just at the end | `campaign_B_trajectories.parquet` | 113 positions × 6 numbers |
| 4 | the bunch's **full six-dimensional shape** at 20 places | `campaign_B_moments.parquet` | 20 planes × (6 means + a 6×6 covariance) |

**Levels 3 and 4 are Campaign B only.** Campaign A is stored at levels 1 and 2 and
nowhere else, so nothing in this pack says what happened *along* the accelerator for
any of its 13,604 runs.

Level 4 is the richest thing here. A bunch is a cloud of electrons in six dimensions — three
positions and three momenta — and its covariance is the 6×6 table of how those six coordinates
vary together. Emittance, beam size, divergence and the energy chirp are all things you can
compute *from* that table; none of them contains it.

**Levels 3 and 4 trade against each other, and they do not cover the same simulations.** Level 3
samples the accelerator finely — 113 positions — but keeps only six derived numbers at each,
so every correlation between coordinates is gone. Level 4 samples it coarsely — 20 planes — and
keeps everything second-order, including the chirp and the position–divergence correlation that
level 3 discards. And level 3 covers **2,855 configurations against level 4's 5,286**, so the
trajectory table is about half the corpus. The cell below prints both counts.

In [3]:
# Level 1 -- the bunch as a few numbers. The units are the trap, so they are printed.
scal = pd.read_parquet(f"{PACK}/campaign_B.parquet",
                       columns=["config", "E_med_MeV", "dE_mad", "q_pC", "q_end", "i_peak",
                                "sigma_z", "n_emit_x", "div_rms"])
r = scal[scal.config == cfg].iloc[0]
print(f"level 1 -- config {cfg} as numbers, each with the unit it is actually in:")
for col, unit, note in [
        ("E_med_MeV", "MeV", "median energy of the bunch"),
        ("dE_mad", "fraction", "energy spread -- a FRACTION, not MeV"),
        ("q_pC", "pC", "bunch charge"),
        ("q_end", "C", "the same charge in coulombs, so 3 pC is 3e-12"),
        ("i_peak", "A", "peak current"),
        ("sigma_z", "m", "bunch length, RMS"),
        ("n_emit_x", "m.rad", "normalised emittance -- how tightly focusable the bunch is"),
        ("div_rms", "rad", "divergence, RMS")]:
    print(f"  {col:12s} {r[col]:>14.6g}  {unit:<9s} {note}")
print()

# Level 2 -- the spectrum, on the grid shared by every row of both campaigns.
sp = pd.read_parquet(f"{PACK}/campaign_B.parquet", columns=["config", "spec_common"])
s = np.asarray(sp[sp.config == cfg].iloc[0].spec_common)   # shape only in this pack version -- see data/README.md
print(f"level 2 -- the spectrum: {s.size} bins over "
      f"{grid[0]:.1f}-{grid[-1]:.1f} MeV, peak at {grid[s.argmax()]:.1f} MeV\n")

level 1 -- config 1163 as numbers, each with the unit it is actually in:
  E_med_MeV           113.582  MeV       median energy of the bunch
  dE_mad             0.102667  fraction  energy spread -- a FRACTION, not MeV
  q_pC                 115.26  pC        bunch charge
  q_end            1.1526e-10  C         the same charge in coulombs, so 3 pC is 3e-12
  i_peak              5834.84  A         peak current
  sigma_z         2.41426e-06  m         bunch length, RMS
  n_emit_x        1.10348e-06  m.rad     normalised emittance -- how tightly focusable the bunch is
  div_rms          0.00100617  rad       divergence, RMS

level 2 -- the spectrum: 200 bins over 25.8-510.3 MeV, peak at 106.1 MeV



In [4]:
# Level 3 -- the beam at 113 positions along the accelerator.
traj = pd.read_parquet(f"{PACK}/campaign_B_trajectories.parquet")
t = traj[traj.config == cfg].sort_values("z_mm")
print(f"level 3 -- {len(t)} positions from {t.z_mm.min():.3f} to {t.z_mm.max():.3f} mm, "
      f"for {traj.config.nunique():,} of the pack's configurations:")
print(t[["z_mm", "E_MeV", "q_pC", "dE_pct", "emit_um", "div_mrad", "sigz_um"]]
      .head(3).to_string(index=False), "\n")

# Level 4 -- the 6-D bunch at 20 planes. `sim_id` is the odd one out; see section 3.
# `group` splits every plane in two: "all" is every stored macroparticle, "cohort" a
# selected sub-population. Pick one and stay with it -- "all" is used throughout this pack,
# and it is the group every answer key is built from.
mom = pd.read_parquet(f"{PACK}/campaign_B_moments.parquet")
mom["config"] = mom.sim_id.str.removeprefix("Config_").astype(int)
m = mom[(mom.config == cfg) & (mom.group == "all")].sort_values("plane_mm")
cov_cols = [c for c in mom.columns if c.startswith("cov_")]
print(f"level 4 -- {len(m)} planes from {m.plane_mm.min():.2f} to {m.plane_mm.max():.2f} mm, "
      f"6 means and {len(cov_cols)} covariance entries each, "
      f"for {mom[mom.group == 'all'].config.nunique():,} configurations "
      f"(group values: {', '.join(sorted(mom.group.unique()))})")
print("  the 6 coordinates are (zeta, y, z, ux, uy, uz):",
      "zeta along the bunch in um, y and z across it in um, and the three momenta")
row = m.iloc[len(m) // 2]
C = np.zeros((6, 6))
for i in range(6):
    for j in range(i, 6):
        C[i, j] = C[j, i] = row[f"cov_{i}{j}"]
print(f"\n  the covariance at plane {row.plane_mm:.2f} mm, as the 6x6 matrix it is:")
print(np.array2string(C, precision=3, suppress_small=False))

level 3 -- 113 positions from 2.015 to 7.054 mm, for 2,455 of the pack's configurations:
    z_mm     E_MeV       q_pC   dE_pct   emit_um  div_mrad  sigz_um
2.015055 42.049046 133.855072 8.952054 11.427250 26.040369 3.539988
2.060053 45.188580 133.854492 8.831645 12.880788 24.394506 3.607092
2.105055 48.405617 133.854477 8.774195 14.642128 22.732237 3.675391 



level 4 -- 20 planes from 2.00 to 7.00 mm, 6 means and 21 covariance entries each, for 4,886 configurations (group values: all, cohort)
  the 6 coordinates are (zeta, y, z, ux, uy, uz): zeta along the bunch in um, y and z across it in um, and the three momenta

  the covariance at plane 3.00 mm, as the 6x6 matrix it is:
[[ 1.497e+01 -2.344e+00  2.061e+00 -3.867e+01 -4.046e-02  3.243e-01]
 [-2.344e+00  8.005e+02  2.083e+01 -4.209e+01  7.451e+01  1.431e+00]
 [ 2.061e+00  2.083e+01  3.041e+02  2.136e+01  1.431e+00  3.075e+01]
 [-3.867e+01 -4.209e+01  2.136e+01  9.661e+03  2.352e-01 -1.777e+00]
 [-4.046e-02  7.451e+01  1.431e+00  2.352e-01  1.314e+01  1.298e-01]
 [ 3.243e-01  1.431e+00  3.075e+01 -1.777e+00  1.298e-01  4.785e+00]]


### What is **not** in the pack, and it matters

- **The electrons themselves.** Level 4 is the bunch's mean and covariance, not its particles.
  The simulations do track individual electrons; those clouds are a separate download,
  level 5 (`level5/`, 4.7 GB for the 20 planes, about 50 GB in full).
- **The laser and the plasma fields.** Nothing here says what the laser was doing *inside*
  the cell — only what was set going in and what came out. The machine between them is a black
  box in this pack.
- **The answer keys**, and the 714 Campaign B and 1,915 Campaign A configurations behind them.
  See section 5.

## 2. Every file, opened

The loader above opens two of these. Here is all of them, with the real shape of each
rather than a description of it.

In [5]:
def describe(path):
    """One line on what a pack file actually holds, read from the file itself."""
    if path.endswith(".parquet"):
        f = pq.ParquetFile(path)
        return f"{f.metadata.num_rows:>9,} rows x {len(f.schema_arrow.names):>2} columns"
    if path.endswith(".npy"):
        return f"array of {np.load(path).shape[0]:,}"
    if path.endswith(".csv"):
        d = pd.read_csv(path, nrows=None)
        return f"{len(d):>9,} rows x {d.shape[1]:>2} columns"
    if path.endswith(".json"):
        d = json.load(open(path))
        if isinstance(d, dict):
            return "keys: " + ", ".join(list(d)[:6])
        return f"a list of {len(d)}"
    return ""

for name in sorted(os.listdir(PACK)):
    path = os.path.join(PACK, name)
    if not os.path.isfile(path):
        continue
    print(f"{name:38s} {os.path.getsize(path) / 1e6:7.1f} MB   {describe(path)}")

DATA_CARD.md                               0.0 MB   
campaign_A.parquet                        27.5 MB      13,604 rows x 37 columns
campaign_B.parquet                        87.2 MB       4,886 rows x 38 columns
campaign_B_moments.parquet                62.7 MB     195,360 rows x 57 columns
campaign_B_trajectories.parquet           10.0 MB     277,415 rows x 13 columns
common_energy_grid_MeV.npy                 0.0 MB   array of 200
pack_reference.json                        0.2 MB   keys: what, key, nothing_is_enforced, sections, scores
test_direct.parquet                        0.0 MB         714 rows x  6 columns
test_inverse.parquet                       0.2 MB       1,915 rows x  9 columns
test_moments_planes.parquet                0.0 MB       6,426 rows x  7 columns
test_trajectory.parquet                    0.0 MB      80,682 rows x  7 columns
test_trajectory_ood.parquet                0.0 MB      60,568 rows x  7 columns


### And what each one is for

| file | one sentence |
|---|---|
| `campaign_A.parquet` | Campaign A's finished bunches — four knobs in, the bunch out, one row per simulation. Holds **both** of Campaign A's scans; the `scan` column says which. |
| `campaign_B.parquet` | the same for Campaign B, with the density-profile knobs and the profile curve itself. |
| `campaign_B_trajectories.parquet` | the beam at 113 positions along the accelerator, for the Campaign B configurations that have one (count printed above). |
| `campaign_B_moments.parquet` | the bunch's six means and full 6×6 covariance at 20 planes. |
| `common_energy_grid_MeV.npy` | the one energy axis every `spec_common` column is on, 200 bins. |
| `pack_reference.json` | **one file, five sections**, all keyed on `config`: `reference_split` (the frozen split the internal reference numbers were scored on), `participant_split` (the fold you can score yourself on, and the pool you draw your data-budget subsets from), `ood_split` (train below a density threshold, score above it), `participant_ood_split` (the same axis on ids you can score), `plane_holdout` (train on 11 planes, predict 9). |
| `test_*.parquet` | the five hidden-test input files. **No target column appears in any of them.** |

The `test_*.parquet` files are what a submission is built from: inputs only, one row per prediction
the organisers will score.

## 3. How the files join, and the two traps in doing it

**`config` is one simulation, and it is the key everywhere.** One row of `campaign_A.parquet`
or `campaign_B.parquet` is one simulation, `config` names it, it is unique in both files, and
all five test files join on it. So there is nothing to group and nothing to be careful about:
a split on `config` is a split on rows.

Two columns ride along for traceability and are **not** keys. `draw` is the index the source
scan gave the run, and `sub_scan` says which of Campaign A's five stacked random sub-scans it
came from; together they locate the run's directory on the machine that produced it. `draw`
repeats once per sub-scan, and the five runs sharing one are **five different settings**, not
one setting five times — the cell below prints such a group so you can see it. `row_id` is the
same information as a readable string, `scan:draw:sub_scan`.

Between sub-scans 0 and 1 only `a_0` is repeated: `p_1` differs by up to 10 mbar, `c_N2`
across its whole range, and `x_of` by 300 µm.

**Trap 1 — the moments table spells the id differently.** It is the string `"Config_1001"` in
`sim_id`; everywhere else, `test_moments_planes.parquet` included, the same id is the integer
`1001`. Strip the prefix before joining.

**Trap 2 — in the moments columns, `z` is a transverse coordinate.** The position along the
accelerator there is `plane_mm`. In the trajectory table the position along the accelerator is
`z_mm`. Two files, two meanings of the letter z.

In [6]:
print("campaign_A -- one row is one simulation, and config is unique in both scans:")
print(A.groupby("scan").agg(rows=("config", "size"),
                            configs=("config", "nunique"),
                            lowest=("config", "min"),
                            highest=("config", "max")).to_string(), "\n")

print("the five random-scan runs that share draw 996 are five different settings:")
print(A[(A.scan == "random") & (A.draw == 996)]
      [["config", "draw", "sub_scan", "row_id"] + A_KNOBS].to_string(index=False), "\n")

print("trap 1 -- strip the prefix before joining:")
print(f"  sim_id {mom.sim_id.iloc[0]!r}  ->  config {int(mom.config.iloc[0])}\n")

print("trap 2 -- plane_mm is the position along the accelerator; the z in these column")
print("          names is across the bunch. The knobs are already in this table, so join")
print("          it to the endpoint table when you want the finished bunch beside it:")
joined = m.merge(scal[["config", "E_med_MeV", "i_peak"]], on="config")
print(joined[["plane_mm", "mean_zeta_um", "mean_z_um", "mean_uz", "i_peak"]]
      .head(3).to_string(index=False))
print(f"\n  {joined.shape[0]} planes x {joined.shape[1]} columns; "
      f"the knobs {', '.join(B_KNOBS)} were already there")

campaign_A -- one row is one simulation, and config is unique in both scans:
         rows  configs  lowest  highest
scan                                   
random   9816     9816       2    12005
uniform  3788     3788  100000   103999 

the five random-scan runs that share draw 996 are five different settings:
 config  draw  sub_scan         row_id       p_1      a_0     c_N2        x_of
    997   996         0 A_random:996:0 49.528177 1.377227 0.081421  616.700440
   3398   996         1 A_random:996:1 45.136158 1.377227 0.015096  916.700440
   5799   996         2 A_random:996:2 11.425069 1.418156 0.063777  859.551954
   8200   996         3 A_random:996:3 92.065347 1.132058 0.036473 1045.944341
  10601   996         4 A_random:996:4 18.863105 1.202265 0.031782    3.428994 

trap 1 -- strip the prefix before joining:
  sim_id 'Config_0'  ->  config 0

trap 2 -- plane_mm is the position along the accelerator; the z in these column
          names is across the bunch. The knobs are a

### One injection flag, and the charge unit

`injected` is `q_pC >= 3.0`, one rule computed identically for all three campaigns. The source
files' own labels are not shipped, so there is only ever one definition in play.

**`q_end` is in coulombs**, so a 3 pC cut is `3e-12`, not `3`; `q_pC` beside it is the same
number in picocoulombs.


## 4. The splits — how the reference numbers were made

**Nothing here is enforced.** Nobody checks what you trained on, and you cannot train on the
hidden test in any case, because its answers are not in the pack. This JSON file is a
record of how our internal reference numbers were produced, so that your number can be put
beside them.

That matters most for the questions that are *about* what you trained on. "Does accuracy still
climb when you halve the training data" is answered by running your own model at several sizes,
on subsets of the pool that you choose yourself — no fixed lists ship, and how you pick them is
part of the question. "How far
does it extrapolate" is answered by holding out high-density configurations — and those are
public, with their answers, so if you train on them the number measures nothing. Draw your own
line or reuse ours; the only thing you lose by ignoring these files is comparability.

Each file states its own purpose in a `what` key. Here they are, read from the files.

In [7]:
for name in sorted(n for n in os.listdir(PACK) if n.endswith(".json")):
    d = json.load(open(os.path.join(PACK, name)))
    if isinstance(d, dict) and "what" in d:
        print(f"{name}\n    {d['what']}\n")

pack_reference.json
    everything you need to make your number comparable to the published ones: the id lists each axis is defined on, and the scores themselves



**Which to use.** If you want a number you can compare with our internal references, train
on the `reference_split` section's train ids that are still public (400 of them sit in the hidden
out-of-distribution test) and read the references as a target — you cannot
reproduce them, because they are scored on a test fold that is not in the public tables. If you
want a number you can compute yourself today, use the `participant_split` section.

## 5. What is not here, and why

**714 Campaign B and 1,915 Campaign A simulations are absent.** They are the hidden test set.
Every shipped test file's targets used to be recoverable from a public table with a one-line
join, so the simulations behind the five answer keys left the public tables entirely. The cut
is **exactly the tested simulations** — one config, one simulation, one row — and Campaign A's
is drawn from the runs that injected, because a run that produced no bunch is nothing to invert
and stays public.

**273 Campaign A runs were deleted, not flagged.** Their simulated laser envelope oscillates in
a way the physics does not allow, which means the run is a numerical failure rather than an
unusual machine setting. They are simply absent: no column, no list, nothing to filter. A
repaired envelope would have manufactured physics the simulation never produced.

**Three things were harmonised before you see them**, all described in `DATA_CARD.md`: the two
campaigns stored energy in different units, every spectrum was on its own energy axis, and the
two Campaign A scans had been post-processed under different injection rules. Energies are MeV
everywhere, spectra are on one shared grid, and `injected` is one rule — `q_pC >= 3.0` —
computed identically for everything.


## 6. The columns you will actually use

Units are what people get wrong here, so every column states one. **All energies are MeV**;
there is no gamma column and no unit flag to check.

### The ids

| column | meaning |
|---|---|
| `scan` | which Campaign A scan a row came from: `random` or `uniform` |
| `config` | **the simulation.** Unique in both campaigns' files, and what every test file and every submission is keyed by |
| `draw` | the index the source scan gave the run. It repeats across the random scan's sub-scans, so it is not a key |
| `sub_scan` | which of the random scan's five stacked sub-scans a row came from |
| `row_id` | the same provenance as a readable string, `scan:draw:sub_scan` |

### The knobs (what the operator sets)

| column | meaning | unit | campaign |
|---|---|---|---|
| `p_1` | injection-region pressure | mbar | A |
| `a_0` | laser amplitude (normalised vector potential) | dimensionless | A, B (fixed in B) |
| `c_N2` | nitrogen concentration | fraction | A |
| `x_of` | laser focal position | µm | A, B |
| `P_max` | peak pressure of the density profile | Pa | B |
| `cN2_max` | peak nitrogen fraction | fraction | B |
| `L_inj` | length of the injection region | mm | B |
| `dip_frac` | depth of the density dip | fraction | B |

### The beam (what comes out)

| column | meaning | unit |
|---|---|---|
| `E_med_MeV` | median beam energy | MeV |
| `E_mean_MeV` | mean beam energy | MeV |
| `dE_mad` | energy spread, median absolute deviation | **fraction of the energy**, not MeV |
| `dE_std` | energy spread, standard deviation | fraction |
| `q_end` | bunch charge | **coulombs** (so 3 pC is `3e-12`) |
| `i_peak` | peak current | A |
| `sigma_z`, `sigma_y` | bunch length and transverse size, RMS | m |
| `n_emit_x`, `n_emit_y` | normalised RMS emittance | m·rad |
| `div_rms` | RMS divergence | rad |
| `beta_x`, `alpha_x` | Twiss parameters | m, — |
| `spec`, `ener_axis_MeV` | the energy spectrum and its own axis | pC per MeV, MeV |
| `spec_common` | the same spectrum on the shared grid | shape only in this pack version: × the row's native bin width gives pC per bin |
| `x_p`, `n_e_p` | the gas density profile | **µm**, m⁻³ |

In [8]:
beam = ["E_med_MeV", "dE_mad", "q_end", "i_peak", "sigma_z", "n_emit_x", "div_rms"]
B.loc[B.injected, beam].describe().T[["min", "50%", "max"]]

,min,50%,max
E_med_MeV,6.750084e+01,1.319481e+02,2.412779e+02
dE_mad,1.902867e-02,7.409444e-02,5.084738e-01
q_end,3.037840e-12,6.134964e-11,2.644033e-10
i_peak,1.878321e+02,3.727876e+03,8.859145e+03
sigma_z,8.179158e-07,2.095513e-06,7.954796e-06
n_emit_x,1.809471e-07,4.983905e-07,8.157186e-06
div_rms,4.642461e-04,7.737146e-04,3.673690e-03


## 7. The spectra — one shared grid, and one correction in this pack version

Section 5 lists what the pack harmonised. For the spectra that means every row's `spec_common`
sits on the one 200-bin grid in `common_energy_grid_MeV.npy`, while its native spectrum keeps its
own axis in `ener_axis_MeV`.

**In this pack version `spec_common` is not yet charge per bin.** Each row is off by one constant
factor, the width of that row's own native energy bin, so the shape is right and the total is
not. Multiply by `np.diff(ener_axis_MeV).mean()` of the same row and the spectrum sums to `q_pC`;
the cell below does it on one row. The next pack version ships it corrected.


In [9]:
print("the shared grid:", f"{grid[0]:.1f} to {grid[-1]:.1f} MeV in {len(grid)} bins")
spec = pd.read_parquet(f"{PACK}/campaign_B.parquet",
                       columns=["injected", "q_pC", "ener_axis_MeV", "spec_common"])
row = spec[spec.injected].iloc[0]
print("one row's native axis runs", f"{np.min(row.ener_axis_MeV):.1f} to "
      f"{np.max(row.ener_axis_MeV):.1f} MeV -- its own, not the shared one")
charge_per_bin = row.spec_common * np.diff(row.ener_axis_MeV).mean()   # pC in each shared bin
print(f"as shipped the row sums to {np.sum(row.spec_common):.2f}; corrected, "
      f"{charge_per_bin.sum():.2f} pC against q_pC = {row.q_pC:.2f} pC")


the shared grid: 25.8 to 510.3 MeV in 200 bins
one row's native axis runs 27.5 to 486.6 MeV -- its own, not the shared one
as shipped the row sums to 49.96; corrected, 115.26 pC against q_pC = 115.26 pC


## 8. The task — the beam read back to the settings

$$(\texttt{E\_med\_MeV},\ \texttt{dE\_mad},\ \texttt{q\_end},\ \texttt{i\_peak},\
\texttt{sigma\_z},\ \texttt{div\_rms},\ \texttt{n\_emit\_x},\ \texttt{beta\_x})
\;\longrightarrow\;(\texttt{p\_1},\ \texttt{a\_0},\ \texttt{c\_N2},\ \texttt{x\_of})$$

Campaign A's random scan, injected rows.

**Submit** a CSV with one row per test simulation: `config`, then your predicted `p_1`, `a_0`,
`c_N2` and `x_of`. Scored by R² per setting; the reference mean is over the first three.

**The cell below is setup.** It defines the `train` and `test` frames every later cell uses.
A plain random split is enough: one row is one simulation, so nothing spans the two sides.


In [10]:
from sklearn.model_selection import train_test_split

# The task below is the random scan's: the hidden test set is drawn from it.
inj_a = A[(A.scan == "random") & A.injected]
train, test = train_test_split(inj_a, test_size=0.2, random_state=0)
print(f"train {len(train):,} rows / {train.config.nunique():,} configs"
      f"   test {len(test):,} rows / {test.config.nunique():,} configs")
print(f"configs in both sides: {len(set(train.config) & set(test.config))}")


train 6,126 rows / 6,126 configs   test 1,532 rows / 1,532 configs
configs in both sides: 0


In [11]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Four knobs in, four knobs out. Our reference ensemble predicts only the first three --
# it was built before x_of was asked for -- so the comparison below is on those three.
TARGETS = ["p_1", "a_0", "c_N2", "x_of"]
FEATURES = ["E_med_MeV", "dE_mad", "q_end", "i_peak", "sigma_z", "div_rms",
            "n_emit_x", "beta_x"]

data_tr = train.dropna(subset=FEATURES + TARGETS)
data_te = test.dropna(subset=FEATURES + TARGETS)

def design(frame):
    x = frame[FEATURES].copy()
    for col in ["q_end", "i_peak", "n_emit_x"]:      # these span decades
        x[col] = np.log10(np.clip(x[col], 1e-30, None))
    return x.to_numpy()

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(design(data_tr), data_tr[TARGETS].to_numpy())
pred = model.predict(design(data_te))

dumb = DummyRegressor(strategy="mean").fit(
    design(data_tr), data_tr[TARGETS].to_numpy()).predict(design(data_te))

for i, t in enumerate(TARGETS):
    print(f"  {t:6s}  linear R2 {r2_score(data_te[t], pred[:, i]):+.4f}"
          f"   mean-predictor {r2_score(data_te[t], dumb[:, i]):+.4f}")
print(f"\n  mean linear R2 "
      f"{r2_score(data_te[TARGETS], pred, multioutput='uniform_average'):+.4f}")

  p_1     linear R2 +0.8512   mean-predictor -0.0000
  a_0     linear R2 +0.3173   mean-predictor -0.0015
  c_N2    linear R2 +0.7809   mean-predictor -0.0001
  x_of    linear R2 +0.1629   mean-predictor -0.0015

  mean linear R2 +0.5281


### Score it, then submit it

`pallas_score.py` beside this notebook is the scorer the organisers run. Score yourself on
your own held-out rows first — you have the answers for those. Then predict the **hidden test
set** (`test_inverse.parquet`, inputs only) and submit that file.

In [12]:
import sys
sys.path.insert(0, ".")
from pallas_score import score_submission

# Your own held-out rows, where you DO have the answers.
own = pd.DataFrame(pred, columns=TARGETS)
own.insert(0, "config", data_te.config.to_numpy())
own_key = data_te[["config"] + TARGETS]
print({k: (round(v, 4) if isinstance(v, float) else v)
       for k, v in score_submission(own, own_key).items()})

{'p_1': 0.8512, 'a_0': 0.3173, 'c_N2': 0.7809, 'x_of': 0.1629, 'mean': 0.5281, 'n_rows': 1532}


In [13]:
# The hidden test set: inputs only, no targets. Predict and write.
hidden = pd.read_parquet(f"{PACK}/test_inverse.parquet")
hidden_pred = model.predict(design(hidden))
submission = pd.DataFrame(hidden_pred, columns=TARGETS)
submission.insert(0, "config", hidden.config.to_numpy())
submission.to_csv("submission_inverse.csv", index=False)
print(f"wrote submission_inverse.csv  ({len(submission):,} rows)")
submission.head()

wrote submission_inverse.csv  (1,915 rows)


,config,p_1,a_0,c_N2,x_of
0,1,64.017818,1.310104,0.100282,623.839307
1,1012,59.448577,1.411030,0.124016,819.147505
2,1019,69.605175,1.428446,0.023468,1112.522479
3,1034,64.293746,1.337104,-0.008826,1098.082401
4,1035,34.436233,1.321720,0.049904,892.601820


### Only now: where that score sits

Try the task before reading this. Our mixture-density ensemble (in review, not yet published) reaches **mean R²
0.9100** and the **oracle ceiling is 0.9909** (both in `pack_reference.json`). The ceiling is
below 1 because the map is not injective — different settings give similar beams — so that gap
is degeneracy, not model error, and `a_0` is the hardest column for every model including ours.
A point estimate is therefore the wrong object; see `README.md`.

**Those two rows cover `p_1`, `a_0` and `c_N2` only.** Our model was built before
`x_of` was a target, so nobody has a reference number for the focal position — your score on it
is the first one. Compare your three-target mean with theirs; read `x_of` on its own.

In [14]:
REFERENCE = {"p_1": 0.9563, "a_0": 0.8264, "c_N2": 0.9473}
ORACLE = {"p_1": 0.9940, "a_0": 0.9858, "c_N2": 0.9928}
PUBLISHED = list(REFERENCE)          # the three the reference model predicts
yours = {t: r2_score(data_te[t], pred[:, i]) for i, t in enumerate(TARGETS)}
table = pd.DataFrame({"your baseline": yours,
                      "our ensemble (in review)": REFERENCE,
                      "oracle ceiling": ORACLE}).T
# the mean is over the PUBLISHED three, so the three rows are comparable; x_of stands alone
table["mean (3)"] = table[PUBLISHED].mean(axis=1)
table.round(4)

,p_1,a_0,c_N2,x_of,mean (3)
your baseline,0.8512,0.3173,0.7809,0.1629,0.6498
our ensemble (in review),0.9563,0.8264,0.9473,NaN,0.9100
oracle ceiling,0.9940,0.9858,0.9928,NaN,0.9909


## 9. The other direction — the knobs read forward to the beam

$$(\texttt{P\_max},\ \texttt{cN2\_max},\ \texttt{L\_inj},\ \texttt{dip\_frac},\
\texttt{x\_of})\;\longrightarrow\;(\texttt{E\_med\_MeV},\ \texttt{dE\_mad},\
\texttt{q\_end})$$

Campaign B, injected rows. The same machine read forwards: the settings go in, the finished
bunch comes out. This is the direction our internal PALLAS forward model already does, so there
is a number to chase at the end of it.

**Submit** a CSV with one row per test simulation: `config`, then your predicted median energy,
energy spread and charge. Scored the same way, R² per column.


In [15]:
from pallas_score import to_scored_scale     # the scales the leaderboard scores on

D_KNOBS = ["P_max", "cN2_max", "L_inj", "dip_frac", "x_of"]
D_TARGETS = ["E_med_MeV", "dE_mad", "q_end"]

b_inj = B[B.injected]
b_train, b_test = train_test_split(b_inj, test_size=0.2, random_state=0)

# Fit on the scale the scorer uses, not on the raw numbers: charge spans two decades and
# the spread is a fraction, so a straight fit in physical units would chase the big bunches
# and ignore everything else. `to_scored_scale` is the same function the organisers call.
def scored(frame):
    return np.column_stack([to_scored_scale(t, frame[t]) for t in D_TARGETS])

def physical(pred):
    """Back to the units a submission is written in -- the scorer re-applies the scales."""
    out = pd.DataFrame(pred, columns=D_TARGETS)
    out["dE_mad"] = 10 ** out["dE_mad"] / 100.0      # scored as a percentage, on log10
    out["q_end"] = 10 ** out["q_end"] / 1e12         # scored as log10 of the charge in pC
    return out

direct = make_pipeline(StandardScaler(), LinearRegression())
direct.fit(b_train[D_KNOBS].to_numpy(), scored(b_train))

own_d = physical(direct.predict(b_test[D_KNOBS].to_numpy()))
own_d.insert(0, "config", b_test.config.to_numpy())
print(f"train {len(b_train):,} rows   holdout {len(b_test):,} rows")
print({k: (round(v, 4) if isinstance(v, float) else v)
       for k, v in score_submission(own_d, b_test[["config"] + D_TARGETS]).items()})


train 1,985 rows   holdout 497 rows
{'E_med_MeV': 0.9438, 'dE_mad': 0.2781, 'q_end': 0.8273, 'mean': 0.6831, 'n_rows': 497}


In [16]:
# The hidden test set for this direction: 714 Campaign B settings, no answers.
hidden_d = pd.read_parquet(f"{PACK}/test_direct.parquet")
submission_d = physical(direct.predict(hidden_d[D_KNOBS].to_numpy()))
submission_d.insert(0, "config", hidden_d.config.to_numpy())
submission_d.to_csv("submission_direct.csv", index=False)
print(f"wrote submission_direct.csv  ({len(submission_d):,} rows)")
submission_d.head()


wrote submission_direct.csv  (714 rows)


,config,E_med_MeV,dE_mad,q_end
0,1041,124.931888,0.112132,7.491695e-11
1,1109,127.788142,0.123593,9.952981e-11
2,1067,130.745538,0.074419,7.059407e-11
3,116,125.289689,0.080844,1.126543e-10
4,1076,107.192802,0.095718,2.530695e-10


### And where that one sits

Our internal forward model (not published) reaches **0.9744** on the median energy, **0.8036** on the spread
and **0.9420** on the charge (`pack_reference.json`). It was trained on the organisers' own
fold rather than yours, so read the table below as a target to aim at, not as a like-for-like
race — the gap it shows is real and the energy spread is where it is widest.


In [17]:
DIRECT_REFERENCE = {"E_med_MeV": 0.9744, "dE_mad": 0.8036, "q_end": 0.9420}
yours_d = score_submission(own_d, b_test[["config"] + D_TARGETS])
pd.DataFrame({"your baseline": {t: yours_d[t] for t in D_TARGETS},
              "our internal forward model": DIRECT_REFERENCE}).T.round(4)


,E_med_MeV,dE_mad,q_end
your baseline,0.9438,0.2781,0.8273
our internal forward model,0.9744,0.8036,0.9420


## Where to go next

- **Beat both baselines** — anything non-linear will — then look at *where* each one fails.
- **Report a distribution, not a point**, and score its calibration. Different settings can
  produce nearly the same bunch, which is why even a perfect point predictor stops short on
  the inverse task.
- **Keep splitting on `config`**, so your fold stays comparable with the pack's reference split.
- **`objectives.md`** lists the easy, medium and hard objectives. The two tasks you just ran are
  among the easy ones; the scored trajectory task, the data-budget curve and the two
  out-of-distribution axes are where the challenge actually is.
- **`data/README.md`** gives the splits, the zone rule and the scoring conventions in full.
